## Utils: Array

In [1]:
import numpy as np

In [2]:
HAS_GPU = False
DEVICE = "CPU (Numpy)"
xp = np

try:
    import cupy as _cp
    _ = _cp.zeros((1,))
    _ = _cp.random.randn(1)
    HAS_GPU = True
    xp = _cp
    DEVICE = "GPU (CuPy)"
    _cp.get_default_memory_pool().free_all_blocks()
except Exception:
    HAS_GPU = False
    DEVICE = "CPU (Numpy)"
    xp = np


     cp.array.get() : Returns a copy of the array on host memory

In [3]:

# import cupy as cp
# ar = cp.array([1,2,3,4,6])
# print(ar.device)

# sam = ar.get()
# print(sam.device)

In [4]:
def as_numpy(arr):
    if HAS_GPU and hasattr(arr, "get"):
        return arr.get()
    if isinstance(arr, np.ndarray):
        return arr
    return np.array(arr)

def as_device(arr):
    if HAS_GPU and isinstance(arr, np.ndarray):
        return xp.asarray(arr)
    return arr

def print_device():
    print(f'Using: {DEVICE}')

In [5]:
arr = [1,2,3,4]
arr

[1, 2, 3, 4]

In [6]:
arr = as_numpy(arr)
arr.device

'cpu'

In [7]:
arr = as_device(arr)
arr.device

<CUDA Device 0>

In [8]:
print_device()

Using: GPU (CuPy)


## Data download

In [9]:
#  get data for training
import requests

url = "https://gist.githubusercontent.com/blakesanie/dde3a2b7e698f52f389532b4b52bc254/raw/76fe1b5e9efcf0d2afdfd78b0bfaa737ad0a67d3/shakespeare.txt"
r = requests.get(url)

with open('data.txt', 'wb') as f:
    f.write(r.content)


In [10]:
import requests
from pathlib import Path

def download_data(url):
    file_name = Path('/data.txt')
    if file_name.exists():
        print("Path exists")
    else:
        r = requests.get(url)
        with open('data.txt', 'wb') as f:
            f.write(r.content)

download_data(url)

### Tokenizer

In [1]:
import sys
sys.path.insert(0, 'x:\\VS_CODE\\LLM_track\\CuPy_dev')
from tokenizer_solution.bpe import BPETokenizer, GPT4_SPLIT_PATTERN, GPT4_SPECIAL_TOKENS

model = BPETokenizer(pattern=GPT4_SPLIT_PATTERN)
model


In [2]:
sample_text = '''The Imperial Russian Navy (Russian: Российский императорский флот) operated 
as the navy of the Russian Tsardom and later the Russian Empire from 1696 to 1917.[c] Formally
established in 1696, it lasted until being dissolved in the wake of the February Revolution and
the declaration of the Russian Republic in 1917. It developed from a smaller force that had 
existed prior to Tsar Peter the Great's founding of the modern Russian navy during the Second 
Azov campaign in 1696[3], and expanded in the second half of the 18th century before reaching 
its peak strength by the early part of the 19th century, behind only the British and French f
leets in terms of size.The Imperial Navy drew its officers from the aristocracy of the Empire <|endoftext|>'''


In [3]:
model.pattern

"'(?i:[sdmt]|ll|ve|re)|[^\\r\\n\\p{L}\\p{N}]?+\\p{L}+|\\p{N}{1,3}| ?[^\\s\\p{L}\\p{N}]++[\\r\\n]*|\\s*[\\r\\n]|\\s+(?!\\S)|\\s+"

In [4]:
model.merges

{}

In [5]:
model.vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [6]:

model.load('model\\bpe.model')
model

In [7]:

ids = model.encode(sample_text, allowed_special="all")
print(ids)
print(model.decode(ids))

[321, 333, 288, 323, 334, 446, 58, 568, 581, 587, 41, 590, 32, 10, 320, 258, 335, 264, 258, 288, 592, 297, 593, 258, 288, 367, 338, 32, 398, 54, 285, 32, 343, 55, 344, 99, 93, 595, 10, 101, 272, 399, 108, 345, 257, 100, 273, 32, 398, 54, 44, 325, 597, 600, 601, 605, 273, 258, 608, 264, 258, 613, 404, 297, 10, 116, 257, 615, 264, 258, 288, 620, 273, 32, 343, 55, 46, 405, 621, 338, 294, 625, 627, 468, 347, 32, 10, 101, 120, 412, 267, 631, 285, 455, 633, 258, 634, 348, 636, 264, 258, 638, 288, 335, 375, 258, 639, 32, 10, 65, 122, 328, 644, 273, 32, 398, 54, 91, 51, 645, 297, 648, 273, 258, 649, 475, 264, 258, 32, 378, 329, 381, 419, 652, 32, 10, 269, 115, 654, 657, 658, 258, 477, 479, 264, 258, 32, 307, 329, 381, 44, 661, 480, 258, 665, 297, 667, 290, 10, 341, 115, 273, 670, 264, 672, 46, 321, 333, 323, 675, 351, 425, 338, 258, 677, 264, 258, 367, 32, 100257]
The Imperial Russian Navy (Russian: Российский императорский флот) operated 
as the navy of the Russian Tsardom and later the Russi

In [8]:
model.encode('<|endoftext|>', allowed_special='all')

[100257]